# 03 Train Signal Quality Baselines

This notebook compares two fast, interpretable baselines using the
subject-disjoint split produced in Stage 02. Test data remains untouched.

In [ ]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
AI_SERVICE_ROOT = REPO_ROOT / "apps" / "ai-service"
sys.path.insert(0, str(AI_SERVICE_ROOT))

RAW_DIR = Path(
    os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")
).expanduser().resolve()
PROCESSED_DIR = Path(
    os.environ.get("PURE_PROCESSED_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-v2")
).expanduser().resolve()
MODEL_DIR = Path(os.environ.get("PURE_MODEL_DIR", REPO_ROOT / "models")).expanduser().resolve()

print("Repository:", REPO_ROOT)
print("Raw dataset:", RAW_DIR)
print("Processed dataset:", PROCESSED_DIR)
print("Model directory:", MODEL_DIR)

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, balanced_accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

features_path = PROCESSED_DIR / "features.csv"
if not features_path.exists():
    raise FileNotFoundError("Run notebook 02 or scripts/prepare_but_ppg.py first")
frame = pd.read_csv(features_path)
frame.groupby("split")["label_good_quality"].agg(["count", "mean", "sum"])

In [ ]:
FEATURE_COLUMNS = ['raw_range', 'raw_std', 'derivative_rms', 'derivative_std', 'zero_crossing_rate', 'clipping_ratio', 'skewness', 'excess_kurtosis', 'spectral_entropy', 'dominant_frequency_hz', 'pulse_band_power_ratio', 'peak_count', 'peak_interval_cv']
TARGET = "label_good_quality"

train = frame[frame["split"] == "train"].copy()
validation = frame[frame["split"] == "validation"].copy()
X_train, y_train = train[FEATURE_COLUMNS], train[TARGET].astype(int)
X_validation, y_validation = validation[FEATURE_COLUMNS], validation[TARGET].astype(int)

In [ ]:
models = {
    "logistic_regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=2026)),
    ]),
    "random_forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=400,
            min_samples_leaf=3,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=2026,
        )),
    ]),
}

def binary_metrics(y_true, probability, threshold=0.5):
    prediction = (probability >= threshold).astype(int)
    poor = y_true == 0
    false_acceptance_rate = float(np.mean(prediction[poor] == 1))
    return {
        "precision_good": precision_score(y_true, prediction, zero_division=0),
        "recall_good": recall_score(y_true, prediction, zero_division=0),
        "f1_good": f1_score(y_true, prediction, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "roc_auc": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "false_acceptance_rate_at_0_5": false_acceptance_rate,
    }

results = {}
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_validation)[:, 1]
    results[name] = binary_metrics(y_validation.to_numpy(), probability)
    trained_models[name] = model

results_frame = pd.DataFrame(results).T.sort_values(
    ["false_acceptance_rate_at_0_5", "average_precision"],
    ascending=[True, False],
)
display(results_frame)

In [ ]:
# Selection uses validation data only. Lower false acceptance is prioritized.
selected_model_name = results_frame.index[0]
candidate_directory = MODEL_DIR / "candidates"
candidate_directory.mkdir(parents=True, exist_ok=True)
for name, model in trained_models.items():
    joblib.dump(model, candidate_directory / f"{name}.joblib")

selection = {
    "selected_model": selected_model_name,
    "feature_columns": FEATURE_COLUMNS,
    "seed": 2026,
    "validation_metrics": results,
    "selection_rule": "lowest validation false acceptance rate at 0.5, then highest average precision",
}
MODEL_DIR.mkdir(parents=True, exist_ok=True)
(MODEL_DIR / "model_selection.json").write_text(json.dumps(selection, indent=2) + "\n")
selection